# Local-Level Metrics — From Saved Baselines

**Prerequisite**: Run `local_baseline.ipynb` to generate baseline CSVs.

**Computes** (no SARIMAX re-fitting):
1. Largest drop (within, inflow)
2. Recovery time (within, inflow) — Theil-Sen trend-based
3. Outflow increase — max positive deviation in [landing-3d, landing+6d]
4. Total disruption = |largest_drop| x recovery_days

**Plots**: Relative difference with annotations for every local unit

**Outputs**: `results/local_level/{hurricane}/metrics_{flow}.csv`

In [ ]:
import pandas as pd
import numpy as np
import os
import warnings

import matplotlib.pyplot as plt
from scipy.stats import theilslopes

warnings.filterwarnings('ignore')

LOCAL_ROOT = '../results/local_level'

HURRICANE_CONFIGS = {
    'milton': {
        'landing_date': pd.Timestamp('2024-10-09'),
        'unit_type': 'county',
    },
    'helene': {
        'landing_date': pd.Timestamp('2024-09-26'),
        'unit_type': 'cluster',
    },
}

FLOW_TYPES = ['within', 'inflow', 'outflow']

print('Setup complete.')

In [ ]:
# ── Metric functions (same as regional_metrics.ipynb) ──

def compute_relative_deviation(df):
    eps = 1e-12
    denom = df['y_pred'].replace(0, np.nan) + eps
    rel_diff = (df['y_true'] - df['y_pred']) / denom * 100
    rel_lower = (df['ci_lower'] - df['y_pred']) / denom * 100
    rel_upper = (df['ci_upper'] - df['y_pred']) / denom * 100
    return rel_diff, rel_lower, rel_upper

def compute_largest_drop(rel_diff, landing_date, window_days=6):
    start = landing_date
    end = landing_date + pd.Timedelta(days=window_days)
    window = rel_diff.loc[(rel_diff.index >= start) & (rel_diff.index <= end)]
    if window.empty:
        return None, None
    return float(window.min()), window.idxmin()

def compute_outflow_increase(rel_diff, landing_date, pre_days=3, post_days=6):
    start = landing_date - pd.Timedelta(days=pre_days)
    end = landing_date + pd.Timedelta(days=post_days)
    window = rel_diff.loc[(rel_diff.index >= start) & (rel_diff.index <= end)]
    if window.empty:
        return None, None
    return float(window.max()), window.idxmax()

def trend_based_recovery(rel_diff, landing_date, smooth_window=3, trough_search_days=7):
    rd_smooth = rel_diff.rolling(window=smooth_window, center=True, min_periods=1).mean()
    trough_end = landing_date + pd.Timedelta(days=trough_search_days)
    search = rd_smooth.loc[(rd_smooth.index >= landing_date) & (rd_smooth.index <= trough_end)]
    
    if search.empty or search.min() >= 0:
        return {'trough_date': None, 'recovery_days': None, 'recovery_date': None,
                'slope': None, 'intercept': None, 'rd_smooth': rd_smooth,
                'reason': 'No negative deviation found'}
    
    trough_date = search.idxmin()
    post_trough = rd_smooth.loc[rd_smooth.index >= trough_date]
    vals = post_trough.values
    mono_end = 1
    for i in range(1, len(vals)):
        if vals[i] >= vals[i-1] - 1e-15:
            mono_end = i + 1
        else:
            break
    mono_segment = post_trough.iloc[:mono_end]
    
    if len(mono_segment) < 2:
        return {'trough_date': trough_date, 'recovery_days': None, 'recovery_date': None,
                'slope': None, 'intercept': None, 'rd_smooth': rd_smooth,
                'mono_segment': mono_segment, 'reason': 'Mono segment too short'}
    
    t = np.arange(len(mono_segment), dtype=float)
    slope, intercept, lo_slope, hi_slope = theilslopes(mono_segment.values, t)
    
    if slope <= 0:
        return {'trough_date': trough_date, 'recovery_days': None, 'recovery_date': None,
                'slope': float(slope), 'intercept': float(intercept),
                'rd_smooth': rd_smooth, 'mono_segment': mono_segment,
                'reason': 'Slope <= 0'}
    
    tau_from_trough = -intercept / slope
    recovery_date = trough_date + pd.to_timedelta(tau_from_trough, unit='D')
    days_landing_to_trough = (trough_date - landing_date).days
    recovery_days = days_landing_to_trough + tau_from_trough
    
    return {
        'trough_date': trough_date,
        'recovery_days': float(recovery_days),
        'recovery_date': recovery_date,
        'slope': float(slope),
        'intercept': float(intercept),
        'rd_smooth': rd_smooth,
        'mono_segment': mono_segment,
    }

print('Metric functions defined.')

In [ ]:
# ── Process all baselines ──
for hrc_name, hrc_cfg in HURRICANE_CONFIGS.items():
    landing = hrc_cfg['landing_date']
    hrc_dir = f'{LOCAL_ROOT}/{hrc_name}'
    fig_dir = f'{hrc_dir}/figures'
    os.makedirs(fig_dir, exist_ok=True)
    
    print(f"\n{'='*70}")
    print(f'{hrc_name.upper()} (landing: {landing.date()})')
    print(f"{'='*70}")
    
    for ft in FLOW_TYPES:
        # Find all baseline CSVs for this flow type
        baseline_files = sorted([f for f in os.listdir(hrc_dir)
                                if f.startswith(f'baseline_{ft}_') and f.endswith('.csv')])
        
        if not baseline_files:
            print(f'  {ft}: no baselines found')
            continue
        
        print(f"\n  {'─'*50}")
        print(f'  {ft.upper()}: {len(baseline_files)} units')
        print(f"  {'─'*50}")
        
        metrics_list = []
        
        for bf in baseline_files:
            # Extract unit_id from filename: baseline_{ft}_{uid}.csv
            uid = bf.replace(f'baseline_{ft}_', '').replace('.csv', '')
            
            df = pd.read_csv(f'{hrc_dir}/{bf}', index_col=0, parse_dates=True)
            rel_diff, rel_lower, rel_upper = compute_relative_deviation(df)
            
            row = {'unit_id': uid}
            
            # Largest drop
            drop_val, drop_date = compute_largest_drop(rel_diff, landing)
            row['largest_drop'] = drop_val
            row['drop_date'] = drop_date
            
            # Outflow increase
            inc_val, inc_date = compute_outflow_increase(rel_diff, landing)
            row['largest_increase'] = inc_val
            row['increase_date'] = inc_date
            
            # Recovery (within and inflow)
            if ft in ['within', 'inflow']:
                rec = trend_based_recovery(rel_diff, landing)
                row['recovery_days'] = rec.get('recovery_days')
                row['recovery_date'] = rec.get('recovery_date')
                row['trough_date'] = rec.get('trough_date')
                if rec.get('slope'):
                    row['slope_pct_per_day'] = rec['slope'] * 100
                mono = rec.get('mono_segment')
                rd_smooth = rec.get('rd_smooth')
            else:
                row['recovery_days'] = None
                rec = None
                mono = None
                rd_smooth = None
            
            # Total disruption
            if drop_val is not None and row.get('recovery_days') is not None:
                row['total_disruption'] = abs(drop_val) * row['recovery_days']
            else:
                row['total_disruption'] = None
            
            metrics_list.append(row)
            
            # ── Plot relative difference ──
            fig, ax = plt.subplots(figsize=(14, 5))
            forecast_idx = df.index
            
            ax.plot(forecast_idx, rel_diff.values, 'k-', lw=1.5, label='Relative diff')
            ax.fill_between(forecast_idx, rel_lower, rel_upper,
                           color='red', alpha=0.1, label='95% CI')
            ax.axhline(0, color='gray', ls='--', lw=1)
            ax.axhline(5, color='red', ls='--', lw=1, alpha=0.5)
            ax.axhline(-5, color='red', ls='--', lw=1, alpha=0.5)
            ax.axvline(landing, color='blue', ls='--', lw=2, label='Landing')
            
            if drop_val is not None:
                ax.plot(drop_date, drop_val, 'rv', ms=10, zorder=10, label=f'Drop: {drop_val:.1f}%')
            if ft == 'outflow' and inc_val is not None:
                ax.plot(inc_date, inc_val, 'g^', ms=10, zorder=10, label=f'Increase: {inc_val:.1f}%')
            if rec and rec.get('trough_date'):
                ax.axvline(rec['trough_date'], color='orange', ls='--', lw=1.5, label='Trough')
            if rec and rec.get('recovery_date'):
                ax.axvline(rec['recovery_date'], color='green', ls='--', lw=2,
                          label=f'Recovery: {rec["recovery_days"]:.1f}d')
            if rd_smooth is not None:
                ax.plot(rd_smooth.index, rd_smooth.values, 'b-', lw=1, alpha=0.4)
            if mono is not None and len(mono) >= 2:
                ax.plot(mono.index, mono.values, 'g-', lw=3, alpha=0.5)
            
            ax.set_title(f'{hrc_name.capitalize()} — {uid} — {ft} — Metrics', fontweight='bold')
            ax.set_xlabel('Date')
            ax.set_ylabel('Relative Difference (%)')
            ax.legend(fontsize=8, loc='best')
            ax.grid(True, alpha=0.3)
            plt.tight_layout()
            plt.savefig(f'{fig_dir}/metrics_{ft}_{uid}.png', dpi=100, bbox_inches='tight')
            plt.close()
        
        # Save metrics CSV
        mdf = pd.DataFrame(metrics_list)
        mdf.to_csv(f'{hrc_dir}/metrics_{ft}.csv', index=False)
        
        # Print summary
        valid_drop = mdf['largest_drop'].dropna()
        valid_rec = mdf['recovery_days'].dropna() if 'recovery_days' in mdf else pd.Series()
        print(f'    Drop: [{valid_drop.min():.1f}, {valid_drop.max():.1f}]% (n={len(valid_drop)})')
        if len(valid_rec) > 0:
            print(f'    Recovery: [{valid_rec.min():.1f}, {valid_rec.max():.1f}]d (n={len(valid_rec)}, NA={mdf["recovery_days"].isna().sum()})')
        if 'largest_increase' in mdf:
            valid_inc = mdf['largest_increase'].dropna()
            if len(valid_inc) > 0:
                print(f'    Increase: [{valid_inc.min():.1f}, {valid_inc.max():.1f}]%')
        print(f'    Saved: {hrc_dir}/metrics_{ft}.csv')

In [ ]:
# ── Verification: display metrics tables ──
for hrc_name in HURRICANE_CONFIGS:
    hrc_dir = f'{LOCAL_ROOT}/{hrc_name}'
    print(f"\n{'='*50}")
    print(f'{hrc_name.upper()} METRICS')
    print(f"{'='*50}")
    for ft in FLOW_TYPES:
        path = f'{hrc_dir}/metrics_{ft}.csv'
        if os.path.exists(path):
            mdf = pd.read_csv(path)
            print(f'\n{ft}:')
            cols = ['unit_id', 'largest_drop', 'recovery_days', 'largest_increase', 'total_disruption']
            cols = [c for c in cols if c in mdf.columns]
            display(mdf[cols].round(2))